# Predicción del IAI bajo escenarios de cambio climático

Este notebook predice el Índice de Idoneidad Agrícola (IAI) para los escenarios futuros, utilizando el sistema de modelos LightGBM por región entrenado con el IAI final (versión con subíndice de frío invernal).

**Proceso:**
1. Cargar los modelos LightGBM por cluster y el clustering (scaler + K-Means).
2. Cargar los datos futuros y promediar los dos horizontes (2030 y 2040) por escenario, para reducir la variabilidad interanual.
3. Predecir el IAI de cada parcela y cultivo, asignando primero su región y aplicando el modelo correspondiente.
4. Predecir también el IAI presente (2025) con el mismo modelo, para una comparación limpia (predicho vs predicho).
5. Comparar presente y futuro sobre las mismas parcelas.

**Nota:** el sistema no utiliza el déficit de presión de vapor (VPD), por lo que la predicción futura solo requiere temperatura, precipitación y suelo.

## 1. Configuración

In [9]:
import polars as pl
import numpy as np
import joblib
import json
import os

# ─── RUTAS (ajusta a tu estructura real) ───
MODELS_DIR = "../models"
LGB_DIR    = os.path.join(MODELS_DIR, "lightgbm_final")   # modelos entrenados con el IAI v3
RUTA_FUTURO = "../Data/fut_clean.parquet"                    # datos futuros
RUTA_HIST   = "../Data/hist_iai_v3.parquet"                  # IAI histórico v3 (para comparar)
RUTA_HIST_CLEAN = "../Data/hist_clean.parquet"              # datos históricos limpios (para 2025)

# ─── Variables del modelo (deben coincidir con el entrenamiento) ───
vars_clima = ["tmax", "tmin", "ppt"]                         # sin vpd
vars_suelo = ["ph1to1h2o_r", "awc_r", "profundidad_efectiva_cm",
              "claytotal_r", "dbthirdbar_r", "sandtotal_r", "silttotal_r"]
vars_cluster = ["tmax", "tmin", "ppt", "ph1to1h2o_r", "awc_r",
                "claytotal_r", "dbthirdbar_r", "sandtotal_r", "silttotal_r"]  # las del K-Means

# ─── Cultivos y escenarios ───
FINAL_CROPS = [75, 69, 204, 76, 54, 3, 221, 212, 36, 24, 227, 2]
ESCENARIOS = {"245": "SSP2-4.5", "585": "SSP5-8.5"}

CROP_DICT_EN = {
    75: "Almonds", 69: "Grapes", 204: "Pistachios", 76: "Walnuts",
    54: "Tomatoes", 3: "Rice", 221: "Strawberries", 212: "Oranges",
    36: "Alfalfa", 24: "Wheat", 227: "Lettuce", 2: "Cotton",
}
CROP_ES = {
    75: "Almendras", 69: "Uvas", 204: "Pistachos", 76: "Nueces",
    54: "Tomates", 3: "Arroz", 221: "Fresas", 212: "Naranjas",
    36: "Alfalfa", 24: "Trigo", 227: "Lechuga", 2: "Algodón",
}


## 2. Cargar los modelos y el clustering

In [10]:
# Scaler y K-Means del clustering (no cambian, se obtuvieron de variables biofísicas)
scaler_cluster = joblib.load(os.path.join(MODELS_DIR, "scaler_cluster.pkl"))
kmeans = joblib.load(os.path.join(MODELS_DIR, "kmeans_clusters.pkl"))

# Los 5 modelos LightGBM finales (entrenados con el IAI v3)
modelos = {}
for c in range(5):
    modelo = joblib.load(os.path.join(LGB_DIR, f"modelo_cluster_{c}_lgb.pkl"))
    with open(os.path.join(LGB_DIR, f"features_cluster_{c}.json")) as f:
        feats = json.load(f)
    modelos[c] = {"modelo": modelo, "features": feats}

print(f"K-Means: {kmeans.n_clusters} clusters")
print(f"LightGBM: {len(modelos)} modelos por región")

# VERIFICACIÓN: confirmar que las features son las nuevas (sin vpd, con profundidad)
feats0 = modelos[0]["features"]
print(f"\n¿Usa vpdmean?: {'vpdmean' in feats0}  (debe ser False)")
print(f"¿Usa profundidad?: {'profundidad_efectiva_cm' in feats0}  (debe ser True)")
print(f"Ejemplo de features one-hot: {[f for f in feats0 if f.startswith('crop_name_')][:3]}")


K-Means: 5 clusters
LightGBM: 5 modelos por región

¿Usa vpdmean?: False  (debe ser False)
¿Usa profundidad?: True  (debe ser True)
Ejemplo de features one-hot: ['crop_name_Alfalfa', 'crop_name_Almonds', 'crop_name_Cotton']


## 3. Cargar los datos futuros

In [11]:
df_fut = pl.read_parquet(RUTA_FUTURO)
print(f"Filas futuras: {df_fut.height:,}")
print(f"Años: {sorted(df_fut['year'].unique().to_list())}")

# Verificar que tenga las columnas por escenario
cols_esc = [c for c in df_fut.columns if c.endswith("_245") or c.endswith("_585")]
print(f"\nColumnas por escenario: {cols_esc}")


Filas futuras: 65,830
Años: [2030, 2040]

Columnas por escenario: ['hurs_245', 'hurs_585', 'pr_245', 'pr_585', 'tmax_245', 'tmax_585', 'tmean_245', 'tmean_585', 'tmin_245', 'tmin_585', 'vpdmean_245', 'vpdmean_585']


## 4. Promediar los dos horizontes por escenario

Los años individuales (2030 y 2040) reflejan variabilidad interanual, por lo que se promedian para obtener una señal climática más estable por escenario. El suelo no cambia, se toma constante.

In [12]:
# Variables climáticas con sufijo de escenario (sin vpd)
vars_por_esc = ["tmax", "tmin", "pr", "vpdmean"]  # incluir vpd para el clustering

df_fut_avg = (
    df_fut
    .group_by(["lon", "lat"])
    .agg(
        [pl.col(f"{v}_245").mean().alias(f"{v}_245") for v in vars_por_esc] +
        [pl.col(f"{v}_585").mean().alias(f"{v}_585") for v in vars_por_esc] +
        [pl.col(s).first().alias(s) for s in vars_suelo] +
        [pl.col("crop_id").first().alias("crop_id"),
         pl.col("crop_name").first().alias("crop_name")]
    )
)
print(f"Parcelas tras promediar años: {df_fut_avg.height:,}")

# Verificar que no queden nulos en las variables del modelo
for v in vars_suelo:
    n = df_fut_avg.filter(pl.col(v).is_null()).height
    if n > 0:
        print(f"  Aviso: {n} nulos en {v}")


Parcelas tras promediar años: 32,915


## 5. Funciones de preparación y predicción

In [13]:
def preparar_escenario(df, esc):
    return df.rename({
        f"tmax_{esc}": "tmax",
        f"tmin_{esc}": "tmin",
        f"pr_{esc}": "ppt",
        f"vpdmean_{esc}": "vpdmean",   # para el clustering
    })


def predecir_iai(df_prep, escenario_nombre):
    """Asigna cluster y predice el IAI de cada parcela y cultivo."""
    # Asignar cluster con las variables biofísicas
    X_cl = df_prep.select(vars_cluster).to_numpy()
    X_cl_sc = scaler_cluster.transform(X_cl)
    clusters = kmeans.predict(X_cl_sc)
    df_prep = df_prep.with_columns(pl.Series("cluster", clusters))

    resultados = []
    for crop in FINAL_CROPS:
        col_onehot = f"crop_name_{CROP_DICT_EN[crop]}"
        for c in range(5):
            sub = df_prep.filter(pl.col("cluster") == c)
            if sub.height == 0:
                continue
            feats = modelos[c]["features"]
            X = np.zeros((sub.height, len(feats)), dtype=np.float32)
            for j, f in enumerate(feats):
                if f in sub.columns:
                    X[:, j] = sub[f].to_numpy()
                elif f == col_onehot:
                    X[:, j] = 1.0
            pred = np.clip(modelos[c]["modelo"].predict(X), 0, 1)
            resultados.append(
                sub.select(["lon", "lat"]).with_columns([
                    pl.lit(crop).alias("crop_id"),
                    pl.Series("iai_pred", pred),
                    pl.lit(escenario_nombre).alias("escenario"),
                ])
            )
    return pl.concat(resultados)


## 6. Predecir el IAI futuro para cada escenario

In [14]:
# Variables del K-Means (las 10 originales, CON vpdmean)
vars_cluster = ["tmax", "tmin", "ppt", "vpdmean",
                "ph1to1h2o_r", "awc_r", "claytotal_r",
                "dbthirdbar_r", "sandtotal_r", "silttotal_r"]

In [15]:
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names",
                        category=UserWarning)

todas = []
for esc, nombre in ESCENARIOS.items():
    df_prep = preparar_escenario(df_fut_avg, esc)
    pred = predecir_iai(df_prep, nombre)
    todas.append(pred)
    print(f"{nombre}: {pred.height:,} predicciones")

iai_futuro = pl.concat(todas)
print(f"\nTotal: {iai_futuro.height:,} predicciones futuras")


SSP2-4.5: 394,980 predicciones
SSP5-8.5: 394,980 predicciones

Total: 789,960 predicciones futuras


## 7. Predecir el IAI presente (2025) con el modelo

Para una comparación limpia se predice también el presente con el mismo modelo, de modo que el sesgo del modelo se cancele al comparar (predicho vs predicho).

In [16]:
df_hist_clean = pl.read_parquet(RUTA_HIST_CLEAN)

# Extraer lon/lat si están en 'location' (formato MongoDB)
if "lon" not in df_hist_clean.columns and "location" in df_hist_clean.columns:
    df_hist_clean = df_hist_clean.with_columns([
        pl.col("location").struct.field("coordinates").list.get(0).alias("lon"),
        pl.col("location").struct.field("coordinates").list.get(1).alias("lat"),
    ])

# Datos de 2025 (base de las proyecciones)
df_2025 = df_hist_clean.filter(pl.col("year") == 2025)
print(f"Parcelas en 2025: {df_2025.height:,}")


def predecir_iai_presente(df_base):
    """Predice el IAI con el modelo sobre datos con nombres base (sin sufijo)."""
    X_cl = df_base.select(vars_cluster).to_numpy()
    X_cl_sc = scaler_cluster.transform(X_cl)
    clusters = kmeans.predict(X_cl_sc)
    df_base = df_base.with_columns(pl.Series("cluster", clusters))

    resultados = []
    for crop in FINAL_CROPS:
        col_onehot = f"crop_name_{CROP_DICT_EN[crop]}"
        for c in range(5):
            sub = df_base.filter(pl.col("cluster") == c)
            if sub.height == 0:
                continue
            feats = modelos[c]["features"]
            X = np.zeros((sub.height, len(feats)), dtype=np.float32)
            for j, f in enumerate(feats):
                if f in sub.columns:
                    X[:, j] = sub[f].to_numpy()
                elif f == col_onehot:
                    X[:, j] = 1.0
            pred = np.clip(modelos[c]["modelo"].predict(X), 0, 1)
            resultados.append(
                sub.select(["lon", "lat"]).with_columns([
                    pl.lit(crop).alias("crop_id"),
                    pl.Series("iai_pred", pred),
                ])
            )
    return pl.concat(resultados)

iai_2025_pred = predecir_iai_presente(df_2025)
print(f"Predicciones 2025: {iai_2025_pred.height:,}")


Parcelas en 2025: 32,915
Predicciones 2025: 394,980


## 8. Comparación presente vs futuro

Se compara el IAI de 2025 (predicho) con el de cada escenario futuro (predicho), por cultivo. Al usar el mismo modelo en ambos, el sesgo del modelo se cancela y el cambio refleja el efecto del clima.

In [17]:
# IAI medio de 2025 predicho por cultivo
base_pred = (
    iai_2025_pred.group_by("crop_id")
    .agg(pl.col("iai_pred").mean().alias("presente"))
)

# IAI medio futuro por cultivo y escenario
fut = (
    iai_futuro.group_by(["crop_id", "escenario"])
    .agg(pl.col("iai_pred").mean().alias("iai"))
)
fut_wide = fut.pivot(values="iai", index="crop_id", on="escenario")

# Tabla comparativa
comparacion = (
    base_pred.join(fut_wide, on="crop_id")
    .with_columns([
        (pl.col("SSP2-4.5") - pl.col("presente")).alias("cambio_245"),
        (pl.col("SSP5-8.5") - pl.col("presente")).alias("cambio_585"),
        pl.col("crop_id").replace_strict(CROP_ES).alias("cultivo"),
    ])
    .select(["cultivo", "presente", "SSP2-4.5", "SSP5-8.5", "cambio_245", "cambio_585"])
    .sort("cambio_585")
    .with_columns(pl.col(pl.Float64).round(3))
)

with pl.Config(tbl_rows=20):
    print(comparacion)


shape: (12, 6)
┌───────────┬──────────┬──────────┬──────────┬────────────┬────────────┐
│ cultivo   ┆ presente ┆ SSP2-4.5 ┆ SSP5-8.5 ┆ cambio_245 ┆ cambio_585 │
│ ---       ┆ ---      ┆ ---      ┆ ---      ┆ ---        ┆ ---        │
│ str       ┆ f64      ┆ f64      ┆ f64      ┆ f64        ┆ f64        │
╞═══════════╪══════════╪══════════╪══════════╪════════════╪════════════╡
│ Trigo     ┆ 0.668    ┆ 0.721    ┆ 0.635    ┆ 0.053      ┆ -0.034     │
│ Arroz     ┆ 0.684    ┆ 0.685    ┆ 0.679    ┆ 0.001      ┆ -0.004     │
│ Nueces    ┆ 0.676    ┆ 0.732    ┆ 0.673    ┆ 0.056      ┆ -0.004     │
│ Naranjas  ┆ 0.68     ┆ 0.716    ┆ 0.686    ┆ 0.036      ┆ 0.006      │
│ Lechuga   ┆ 0.669    ┆ 0.727    ┆ 0.686    ┆ 0.058      ┆ 0.017      │
│ Pistachos ┆ 0.754    ┆ 0.803    ┆ 0.771    ┆ 0.049      ┆ 0.017      │
│ Algodón   ┆ 0.7      ┆ 0.76     ┆ 0.722    ┆ 0.06       ┆ 0.022      │
│ Uvas      ┆ 0.7      ┆ 0.774    ┆ 0.724    ┆ 0.073      ┆ 0.024      │
│ Fresas    ┆ 0.732    ┆ 0.805    ┆ 

**Interpretación esperada:** los cultivos leñosos que dependen del frío invernal (nueces, pistachos, almendras) deberían perder idoneidad en el escenario severo (cambio_585 negativo), mientras que los cultivos anuales de clima cálido deberían mantenerse o mejorar.

In [18]:
# Importancia de tmin en los modelos (el frío depende de ella)
for c in range(5):
    modelo = modelos[c]["modelo"]
    feats = modelos[c]["features"]
    if "tmin" in feats:
        idx = feats.index("tmin")
        imp = modelo.feature_importances_[idx]
        rank = sorted(modelo.feature_importances_, reverse=True).index(imp) + 1
        print(f"Cluster {c}: tmin importancia = {imp}, puesto {rank}")

Cluster 0: tmin importancia = 934, puesto 6
Cluster 1: tmin importancia = 1982, puesto 5
Cluster 2: tmin importancia = 2933, puesto 3
Cluster 3: tmin importancia = 1310, puesto 5
Cluster 4: tmin importancia = 401, puesto 2


## 9. Guardar resultados

In [19]:
iai_futuro.write_parquet("../Data/iai_futuro_predicho_v3.parquet")
iai_2025_pred.write_parquet("../Data/iai_2025_predicho_v3.parquet")
print("Resultados guardados.")


Resultados guardados.
